# 精度验收与性能调优


## 1. 验收方案（在 10.05 基础上新增两组）

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">项</th><th style="text-align:left;">方案</th></tr>
<tr><td>参考实现</td><td>torch：<code>argsort(g, descending=True, stable=True)[:k]</code> → 按 <code>indices</code> gather K/V → <code>F.scaled_dot_product_attention</code>（选中块张量重组）</td></tr>
<tr><td>选择一致性</td><td>先单独验收 sel_idx 与 Host 稳定排序前 k 项 <b>逐位一致</b>（破平规则对齐）——选择不一致时先停止数值验收；即使输出碰巧相同，也不能视为选择正确</td></tr>
<tr><td>数值验收</td><td>逐输出 assert_close（rtol=1e-3/atol=1e-3，RFC 阈值）；ratio ∈ {12.5%, 25%, 50%} 三档 + S ∈ {2048,4096} + H ∈ {64,128} + B=1/B>1 + 无 mask/causal</td></tr>
<tr><td>特殊用例</td><td>全同分、k 边界并列、负分/有限极值、causal 无可见选中 key 的行；合法 ratio 下 k≥1，空选择仅作扩展接口测试</td></tr>
</table>


## 2. 调优维度（在 10.06 基础上新增）

1. **ratio 扫描**：12.5%/25%/50% 三档的耗时曲线——验证收益边界（11.04 的估算）；
2. **选择段占比**：先分 Kernel 记录选择与注意力耗时及总耗时；融合后普通 Kernel profiler 行不能直接拆出 Python 段，应结合设备时间线。按占比评估排名/归约/分级选择，不把固定 10% 或“一遍门槛法”当规则；
3. **KV_BLOCK 尺寸（扩展实验）**：RFC 验收固定 128；完成后可比较 {64,128,256}——块越大，选择粒度越粗，候选块和选中块的数量也可能减少。在整除且 k 无取整变化时，固定 ratio 下选中的 token 总数仍是 S×ratio，并不会因块变大而必然增多。变化的是选中内容、块内相关性和循环/搬运开销，哪种尺寸更合适需实测；
4. **段间流水**：g 在所有 Q 行共享，选择通常每次输入只做一次。消费 sel_idx 前必须等待它就绪；若重叠独立的 Q 加载，需单独证明资源与同步关系，不能先消费尚未发布的索引。


## 3. 记录表模板

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">配置</th><th style="text-align:left;">ratio</th><th style="text-align:left;">选择段(ms)</th><th style="text-align:left;">注意力段(ms)</th><th style="text-align:left;">总(ms)</th><th style="text-align:left;">vs 全量 SDPA 加速比</th></tr>
<tr><td>【950 待测】</td><td></td><td></td><td></td><td></td><td></td></tr>
</table>


### 四组定向输入分别检查什么

- 全同分：严格选择块 0…K−1，验证破平顺序与唯一写入。
- 边界并列：如分数按 [-1,0,1] 重复，检查第 K 名附近的同分取舍。
- 分数随块号递增：优先选最后若干块，causal 前缀将没有可见 key，必须严格为 0。
- Q=K=0：每行是其可见选中 V 的均值，同时检查 gather 轴和原始位置 mask。

参考脚本先将完整 selected 与 Host 稳定排序逐位比较，再做数值验收。Golden 对空可见行临时开放一个 key 完成有定义的 SDPA，随后把该行置 0；这一结果来自课程契约，不依赖不同后端对全遮蔽行的默认处理。

`--benchmark` 分别测选择、注意力、组合和全量 SDPA 的预热后 Host 墙钟。组合时间必须实际测量，不能只相加两个独立中位数。全量与稀疏注意力的输出语义不同，这个加速比也包含少算一部分 KV 带来的收益；设备阶段时间另用 profiler 记录。


## 4. 课后练习

1. （实践）实现"选择一致性"独立验收：小规模（num_blocks=32）打印 Kernel sel_idx 与 稳定 argsort 前 k 项，逐位比对脚本。
2. （思考）ratio 固定 25% 时，KV_BLOCK 从 128 变 256：块数减少一半；在整除且 k 无取整变化时，选中的 token 总数是否改变？块内相关性、选中集合和实际耗时又会怎样变化？

In [ ]:
!cat ./answer/11.05_answer.txt

## 5. 本节小结

- 验收分层：**选择一致性（先行、独立）→ 数值 all-close → 特殊用例**；
- 调优新增：ratio 扫描、选择段占比、块尺寸、段间流水；
- 加速比记录表同时记录"计算量节省"与"实际耗时"——两者不等价时去查选择段与访存。